In [1]:
# graphrag_plus.py

import json
import pandas as pd
import networkx as nx

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

import ollama



/Users/iramkamdar/miniconda3/envs/langsearch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load FAQ (already generated from refined.csv using your script)
faq = pd.read_csv("faq.csv", encoding="utf-8")

# Load labels JSONL (intent + entities)
labels = []
with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        labels.append(json.loads(line))

print(f"FAQ rows: {len(faq)}")
print(f"Labelled items: {len(labels)}")


FAQ rows: 13
Labelled items: 111


In [3]:
import json

with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    labels = [json.loads(line) for line in f]

# Extract all unique intents
unique_intents = set()
for item in labels:
    for intent in item.get("labels", {}).get("intents", []):
        unique_intents.add(intent)

print("Unique Intents:", unique_intents)


Unique Intents: {'reschedule', 'follow_up', 'accept_or_decline', 'send_materials', 'request_feedback', 'confirm', 'schedule', 'request_info', 'share_feedback'}


In [4]:
import json

with open("student_email_pairs.labels.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i > 4: break
        print(json.loads(line))


{'id': 'a1bc948c-4475-4b2f-ada8-84de14a52a57', 'labels': {'topic': 'Professor/Academic', 'intents': ['accept_or_decline', 'share_feedback'], 'artifacts': ['offer_letter']}, 'subject': 'Offer for RA Position in Our Lab', 'sender_email': 'jane.smith@university.edu'}
{'id': '406b8fae-4c32-4b7f-bfdf-ba5642bdd466', 'labels': {'topic': 'Group/Event Coordination', 'intents': ['accept_or_decline', 'reschedule', 'request_info'], 'artifacts': []}, 'subject': 'Team Meeting Availability Confirmation', 'sender_email': 'doe.jane@example.com'}
{'id': '9180c14f-f779-4339-8495-eb765e329618', 'labels': {'topic': 'Professor/Academic', 'intents': ['share_feedback', 'request_info', 'request_feedback'], 'artifacts': ['evaluation_sheet', 'application_form', 'draft']}, 'subject': 'Discussion about Recent Submission', 'sender_email': 'drsmith@university.edu'}
{'id': '2bd37763-52d8-4d0e-a872-1208620fc1bc', 'labels': {'topic': 'Feedback & Reviews', 'intents': ['request_info', 'reschedule'], 'artifacts': ['applic

In [5]:
import networkx as nx

G = nx.DiGraph()

for item in labels:
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])

    if not topic and not intents and not artifacts:
        continue

    # Add topic node
    if topic:
        G.add_node(topic, type="topic")

    # Add intents as nodes and edges
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            G.add_edge(topic, intent, relation="HAS_INTENT")

    # Add artifacts as nodes and edges
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            G.add_edge(topic, artifact, relation="USES_ARTIFACT")

print(f"Graph: {len(G.nodes())} nodes, {len(G.edges())} edges")


Graph: 31 nodes, 99 edges


In [6]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

qdrant = QdrantClient(path="qdrant_data")   # persistent local storage

qdrant.recreate_collection(
    collection_name="knowledge_space",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)


/var/folders/qz/d_bbgpmn6cb3v7k9_1nqm_b00000gn/T/ipykernel_13761/706361266.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

## if the above cell doesn't work , RUN rm -f qdrant_data/.lock      

In [ ]:
faq_texts = [
    f"FAQ | Question: {row['question']} | Answer: {row['answer']}"
    for _, row in faq.iterrows()
]
faq_vectors = embedder.encode(faq_texts, show_progress_bar=False)
faq_payloads = []

for idx, row in faq.iterrows():
    faq_payloads.append({
        "type": "faq",
        "id_kind": "faq",
        "faq_id": int(idx),
        "question": row["question"], 
        "answer": row["answer"],
    })


In [8]:
graph_nodes = list(G.nodes(data=True))  # [(name, attrs), ...]

graph_texts = []
graph_payloads = []

for i, (name, attrs) in enumerate(graph_nodes):
    ntype = attrs.get("type", "unknown")
    neighbors = list(G.successors(name)) + list(G.predecessors(name))
    neighbors_str = ", ".join(neighbors) if neighbors else "None"

    text = f"GRAPH_NODE | Type: {ntype} | Name: {name} | Neighbors: {neighbors_str}"
    graph_texts.append(text)

    graph_payloads.append({
        "type": "graph_node",
        "id_kind": "graph_node",
        "node_name": name,
        "node_type": ntype,
        "neighbors": neighbors,
    })

graph_vectors = embedder.encode(graph_texts, show_progress_bar=False)


In [9]:
points = []

# FAQ points
for i, (vec, payload) in enumerate(zip(faq_vectors, faq_payloads)):
    points.append(
        PointStruct(
            id=i,
            vector=vec.tolist(),
            payload=payload
        )
    )

offset = len(points)

# Graph-node points (id continues after FAQ)
for j, (vec, payload) in enumerate(zip(graph_vectors, graph_payloads)):
    points.append(
        PointStruct(
            id=offset + j,
            vector=vec.tolist(),
            payload=payload
        )
    )

qdrant.upsert(collection_name="knowledge_space", points=points)
print(f"✅ Upserted {len(points)} total points into 'knowledge_space'")


✅ Upserted 44 total points into 'knowledge_space'


In [18]:
def simple_intent_classifier(email_text: str) -> str:
    text = email_text.lower()
    if "refund" in text or "charged" in text:
        return "Refund Request"
    if "meeting" in text or "call" in text:
        return "Meeting Request"
    return "General Inquiry"


In [10]:
import ollama
import json

def classify_intent_llm(email_text: str, available_intents: list):
    """
    Use LLM (Ollama) to classify the intent based on email text and your labeled intent list.
    """
    prompt = f"""
    You are an intent classification assistant.
    Below is a list of valid intents extracted from training data:
    {available_intents}

    Read the email carefully and return ONLY a JSON object with the most relevant intent.
    If multiple intents fit, return the one that best describes the user's main goal.

    Email:
    \"\"\"{email_text}\"\"\"

    Output example:
    {{"intent": "request_feedback"}}
    """

    response = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )

    try:
        content = response["message"]["content"]
        parsed = json.loads(content)
        return parsed.get("intent", "general_inquiry")
    except Exception:
        return "general_inquiry"


In [17]:
def build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info):
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"

    graph_section = "\n".join([
        f"{i+1}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}"
        for i, g in enumerate(graph_hits)
    ]) or "None"

    expansion_section = "\n".join([
        f"{i+1}. {node} → {neighbors}"
        for i, (node, neighbors) in enumerate(expanded_graph_info.items())
    ]) or "None"

    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intent**: {intent}

📘 **Relevant FAQs**
{faq_section}

🧩 **Graph Context**
{graph_section}

🔗 **Related Concepts**
{expansion_section}

---

Write your reply in Zubair’s tone:
- Acknowledge the sender and context.
- If an action is requested, confirm or ask a polite follow-up question.
- Keep the reply under 120 words.
- Do NOT invent facts — only use what’s in context.
"""
    return prompt


In [ ]:
"""
def build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info):
    faq_section = ""
    for i, f in enumerate(faq_hits, 1):
        faq_section += f"{i}. Q: {f['question']}\n   A: {f['answer']}\n"

    graph_section = ""
    for i, g in enumerate(graph_hits, 1):
        graph_section += f"{i}. Node: {g['node_name']} (type={g['node_type']}), neighbors={g.get('neighbors', [])}\n"

    expansion_section = ""
    for i, (node, neighbors) in enumerate(expanded_graph_info.items(), 1):
        expansion_section += f"{i}. {node} → {neighbors}\n"

    prompt = f
You are a helpful student copilot.

USER EMAIL:
\"\"\"{email_text}\"\"\"

DETECTED INTENT: {intent}

FACTUAL CONTEXT (from FAQ):
{faq_section if faq_section else "None"}

GRAPH CONTEXT (graph nodes hit by retrieval):
{graph_section if graph_section else "None"}

GRAPH EXPANSION (neighbors of important nodes):
{expansion_section if expansion_section else "None"}

Using ONLY the information above, write a polite, accurate, and concise reply to the user.
If some detail is not present, do NOT invent facts.

    return prompt

"""

In [ ]:
def answer_email(email_text: str, top_k: int = 6, show_context: bool = True):
    # Intent classification (placeholder)
    intent = classify_intent_llm(email_text, list(unique_intents))


    # Embed query
    q_vec = embedder.encode([email_text])[0].tolist()

    # Qdrant unified retrieval
    hits = qdrant.query_points(
    collection_name="knowledge_space",
    query=q_vec,   
    limit=top_k
).points


    faq_hits, graph_hits = [], []

    # Separate by type
    for h in hits:
        p = h.payload
        if p["type"] == "faq":
            faq_hits.append({"score": h.score, **p})
        elif p["type"] == "graph_node":
            graph_hits.append({"score": h.score, **p})

    # Graph expansion (for graph hits)
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Print context (for debug)
    if show_context:
        print("\n🔍 Retrieved FAQ Chunks:")
        if faq_hits:
            for f in faq_hits:
                print(f"  [Score {f['score']:.3f}] Q: {f['question']}")
                print(f"                A: {f['answer']}\n")
        else:
            print("  None\n")

        print("Retrieved Graph Nodes:")
        if graph_hits:
            for g in graph_hits:
                print(f"  [Score {g['score']:.3f}] Node: {g['node_name']} (Type: {g['node_type']})")
                print(f"                Neighbors: {g.get('neighbors', [])}\n")
        else:
            print("  None\n")

    # Build prompt
    prompt = build_prompt(email_text, intent, faq_hits, graph_hits, expanded_graph_info)

    # Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Confidence gating
    top_score = hits[0].score if hits else 0.0
    auto_send = top_score > 0.85

    # Return full result
    return {
        "intent": intent,
        "reply": reply_text,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_used": faq_hits,
        "graph_used": graph_hits,
        "graph_expansion": expanded_graph_info,
    }


In [19]:
test_email = (
    "Hi Zubair, I hope you’re doing well. "
    "Our department is hosting a virtual innovation sprint this weekend, and we’d be delighted to have you join."
    "Please take a moment to complete the registration form below so that we can finalize team assignments and logistics."
    "Looking forward to your participation!"
    "Best, Samantha Lee ,Event Coordinator"
)
result = answer_email(test_email)

print("Intent:", result["intent"])
print("Top Score:", result["top_score"])
print("Auto-send:", result["auto_send"])
print("\n------ DRAFT REPLY ------\n")
print(result["reply"])



🔍 Retrieved FAQ Chunks:
  [Score 0.272] Q: What are some of the projects Zubair is currently involved in?
                A: Zubair is actively participating in various projects including a software engineering internship at TechCorp, contributing to an event application form, and working on group presentations with feedback forms. He also engages in discussions about project submissions with Dr. Smith.

  [Score 0.263] Q: How does Zubair respond to invitations for meetings or calls?
                A: Zubair responds promptly to confirm availability and requests calendar invites or Zoom links to lock in the details.

  [Score 0.236] Q: Can you share your resume and portfolio?
                A: Sure, here are the links to my resume and portfolio: Resume - https://drive.google.com/file/d/1aiDXhnJCaFSA8-7Vm_CaUliN6clXI5L_/view?usp=sharing Portfolio - https://zubairatha.vercel.app/

Retrieved Graph Nodes:
  [Score 0.253] Node: calendar_invite (Type: artifact)
                Neighbors: 

### The top similarity score measures how semantically close your input is to your best-matching knowledge chunk. It’s a confidence proxy for deciding whether the LLM’s reply is likely grounded in the right retrieved context.

In [20]:
test_email = (
    "Hi Zubair, I hope you’re doing well. Can I get your linkedin profile to connect?"
)
result = answer_email(test_email)

print("Intent:", result["intent"])
print("Top Score:", result["top_score"])
print("Auto-send:", result["auto_send"])
print("\n------ DRAFT REPLY ------\n")
print(result["reply"])


🔍 Retrieved FAQ Chunks:
  [Score 0.668] Q: Where can I find your LinkedIn profile?
                A: You can view my LinkedIn profile at https://www.linkedin.com/in/zubair-atha/

  [Score 0.549] Q: Where can I find Zubair's professional profiles?
                A: Zubair maintains a LinkedIn profile (https://www.linkedin.com/in/zubair-atha/) and has a GitHub repository (https://github.com/zubairatha) where he shares his work. He also provides a resume via Google Drive.

  [Score 0.481] Q: What additional information has been provided by Zubair for his application?
                A: Zubair has attached previous evaluation forms and feedback reports, a summary of his research experience, and reattached the job description. He also shared his resume and LinkedIn profile link: https://www.linkedin.com/in/zubair-atha/

  [Score 0.345] Q: How can I contact Zubair regarding updates or new sections in a report?
                A: You can reach out to Zubair directly via email to share any 